# Treinar “Hey Ollie” — Ollie

Execute os botões ▶ em ordem. Este notebook evita o erro de dependências no Python 3.13,
usando Python 3.11 separado. O ambiente padrão do Colab não é substituído.

Selecione **Ambiente de execução → Alterar tipo → GPU T4**, se disponível gratuitamente.
Não contrate créditos para este experimento. Se uma etapa falhar, pare nela e envie o erro.
O erro final inclui as últimas linhas da causa. Logs completos ficam em
`/content/ollie-wakeword/logs`, na lateral Arquivos.

O treino usa vozes sintéticas e dados públicos, sem tokens ou gravações pessoais.
Os dados podem ocupar dezenas de GB. O resultado é um primeiro modelo experimental
para uso pessoal não comercial; não comprova o reconhecimento da sua pronúncia.
Baseado no [treinamento oficial openWakeWord](https://github.com/dscripka/openWakeWord#training-new-models)
e no gerador [Piper](https://github.com/rhasspy/piper-sample-generator).


In [ ]:
# @title 1. Preparar ambiente compatível (Python 3.11 separado)
import hashlib, json, sys, urllib.request
from pathlib import Path
ROOT = Path('/content/ollie-wakeword')
ROOT.mkdir(exist_ok=True)
EXPECTED = {'colab_ollie.py': '9580306d3dd2c84c126c85c05a9d4858e17c9358041c2d382f123855656da358', 'execucao.py': '58ae5a36aa128ddb64f36e3a218dc35dac75bc27842f6097c3a6328052f625b3', 'requirements-treino.txt': 'c8ba084f95b4ef5ee5b1a293ae9f4a404869677d9330e0387a482f54ad46169f', 'validar_pipeline.py': '899952a580a24ea4d71e71c0da2ae782ab366742e0db8d25b9ed543549e35723', 'treino_compat.py': '23ee1281776e281a44039c6f8e1150fb5dc09d60d11002a7630fdedd6e27c228'}
BASE = 'https://raw.githubusercontent.com/lucasbado/AssistenteCerebro/main/tools/wakeword/'
for name, expected in EXPECTED.items():
    with urllib.request.urlopen(BASE + name, timeout=60) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError('O código mudou. Reabra a versão atual do notebook antes de continuar.')
    (ROOT / name).write_bytes(content)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Recarrega o helper ao executar novamente após uma atualização do notebook.
sys.modules.pop('execucao', None)
from execucao import run_logged
run_logged([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], log_path=ROOT / 'logs/instalar-uv.log')
PY = ROOT / 'env/bin/python'
if not PY.exists():
    run_logged([sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.11', str(ROOT / 'env')], log_path=ROOT / 'logs/criar-ambiente.log')
def etapa(nome, *argumentos):
    run_logged([str(PY), str(ROOT / 'colab_ollie.py'), nome, *map(str, argumentos)], log_path=ROOT / 'logs' / (nome + '.log'))
etapa('preparar')


In [ ]:
# @title 2. Ouvir a chamada antes de treinar
frase = "hey ollie" # @param {type:"string"}
etapa('previa', '--frase', frase)
from IPython.display import Audio, display
previa = json.loads((ROOT / 'previa.json').read_text())
display(Audio(filename=previa['audio']))
print('Ouça o player. Se o nome sair diferente, experimente hey_oh_lee e execute esta célula novamente.')


Antes de continuar, ouça a prévia. A grafia é apenas uma forma de gerar o som desejado.


In [ ]:
# @title 3. Baixar dados de ambiente e exemplos negativos
etapa('dados')


In [ ]:
# @title 4. Treinar Hey Ollie (só depois de ouvir a prévia)
ouvi_e_aprovei_a_pronuncia = False # @param {type:"boolean"}
exemplos = 1000 # @param {type:"integer"}
passos = 10000 # @param {type:"integer"}
if not ouvi_e_aprovei_a_pronuncia:
    raise RuntimeError('Ouça o áudio da etapa 2 e marque a caixa antes de iniciar o treino.')
etapa('treinar', '--frase', frase, '--exemplos', exemplos, '--passos', passos)


In [ ]:
# @title 5. Baixar modelo ONNX e informações do treino
from google.colab import files
resultado = json.loads((ROOT / 'ultimo_modelo.json').read_text())
print(resultado['validacao'])
files.download(resultado['modelo'])
files.download(str(ROOT / 'ultimo_modelo.json'))


## Depois do download

No PC, abra PowerShell na pasta `D:\Programacao\AssistenteCell` e execute:

```powershell
.\tools\wakeword\ouvir.ps1 -Modelo "$env:USERPROFILE\Downloads\hey_ollie.onnx"
```

Esse teste somente mostra as chamadas detectadas. Ctrl+C encerra.
Depois validaremos pronúncia, falsos disparos e integração com o Android em português.
